In [3]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import ast
import pandas as pd
import json
import numpy as np
from neo4j import GraphDatabase

In [5]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [6]:
uri = "bolt://neo4j-gds-apoc-n10s:7687"
username = "neo4j"
password = "neo4jpassword"

In [ ]:
driver = GraphDatabase.driver(uri, auth=(username, password))

In [16]:
DATA_DIR = "notebooks/rdb"

### neo4j 초기화
- 아래 코드를 통해 넣을 수 없는 데이터(다른 방법으로 이미 넣어둔 데이터)가 있는 경우 아래 코드는 실행하면 안 됨

In [171]:
with driver.session() as session:
    # 모든 관계와 노드 제거
    session.run("MATCH (n) DETACH DELETE n")

### 데이터 로딩
- 스키마 참고
  - https://confluence.tde.sktelecom.com/pages/viewpage.action?pageId=734203873
- 순서
  - (1) PRODUCT
  - (2) PRICE, VOICE, SMS, DATA, TOPUP, CUSTOMERCONDITION, PRODUCT_GROUP, BENEFITCONDITION, DEDUCTIBLE, RELATION

### PRODUCT
- 모바일 요금제 상품
- Label: 요금제

In [172]:
product_table = pd.read_csv(os.path.join(DATA_DIR, "PRODUCT.csv"))

In [173]:
product_table.head(1)

,pmProductID,mappedProductCode,generation,marketingKeyword,productName,productNameInEnglish,lineup,classifiedGroup,productDescription,productSubscriptionCondition,statusOfOperation
0,PA00000001,['NA00007164'],"['LTE generation', '5G generation']","['유심개통', '쓰던폰', 'USIM개통', '비대면', '자급제', '온라인', '데이터100GB 이하', 'T다샵', '다이렉트플랜', '티다이렉트샵', '티다샵', 'wavve할인혜택', '온라인전용요금제', 'T다이렉트전용요금제']",다이렉트5G 38,Direct5G 38,다이렉트플랜,상품 > 기본요금제 > 휴대폰 요금제,월 15GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,운영


In [174]:
eng_colnames = ['pmProductID', 'mappedProductCode', 'generation', 'marketingKeyword', 
                'productName', 'productNameInEnglish', 'lineup', 'classifiedGroup',
                'productDescription', 'productSubscriptionCondition', 'statusOfOperation']

kor_colnames = ['고유ID', '상품코드매핑', '통신규격', '마케팅키워드',
                '상품명', '영문상품명', '라인업', '상품분류',
                '상품설명', '상품가입조건', '운영상태']

kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [175]:
def type_cast(input_data):
    new_data = None
    if pd.isna(input_data):
        input_data = "[]"
    elif '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data = input_data.replace("nan", "")
    
    new_data = ast.literal_eval(input_data)

    return new_data

In [176]:
# 리스트형으로 변환
product_table["generation"] = product_table["generation"].apply(lambda x: type_cast(x))
product_table["marketingKeyword"] = product_table["marketingKeyword"].apply(lambda x: type_cast(x))
product_table["mappedProductCode"] = product_table["mappedProductCode"].apply(lambda x: type_cast(x))

In [177]:
with driver.session() as session:
    for _, row in product_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        session.run(
            """
            CREATE (p:요금제 $props)
            """,
            props=properties
        )

### PRICE
- 상품 가격 정보
- Label: 요금
- 관계
  - 요금제 -[:요금정보]-> 요금

In [178]:
price_table = pd.read_csv(os.path.join(DATA_DIR, "PRICE.csv"))
price_table.head(1)

,pmProductID,monthlyPrice,monthlyPriceWithoutVAT,monthlyPriceWithSelectableInstallment,billingMethod,netPrice
0,PA00000001,38000,34546,38000,후불,34545


In [179]:
eng_colnames = ['monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 
                'billingMethod', 'netPrice']

kor_colnames = ['월정액', '부가세제외월정액', '선택약정할인포함부가세제외월정액', 
                '청구방법', 'net가격']

kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [180]:
with driver.session() as session:
    for _, row in price_table.iterrows():
        # 속성값을 한글 컬럼명으로 변환
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        # MERGE로 동일한 속성값의 '요금' 노드가 없으면 생성, 있으면 재사용
        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (price:요금 {월정액: $월정액, 부가세제외월정액: $부가세제외월정액, 
                               선택약정할인포함부가세제외월정액: $선택약정할인포함부가세제외월정액, 
                               청구방법: $청구방법, net가격: $net가격})
            MERGE (p)-[:요금정보]->(price)
            """,
            product_id=row['pmProductID'],
            월정액=properties['월정액'],
            부가세제외월정액=properties['부가세제외월정액'],
            선택약정할인포함부가세제외월정액=properties['선택약정할인포함부가세제외월정액'],
            청구방법=properties['청구방법'],
            net가격=properties['net가격']
        )

### VOICE
- 음성 제공량 관련 정보
- Label: 음성통화
- 관계
  - 요금제 -[:제공]-> 음성통화

In [181]:
voice_table = pd.read_csv(os.path.join(DATA_DIR, "VOICE.csv"))
voice_table.head(1)

,pmProductID,includedVoiceCall,includedVideoOrValueAddedCall,includedVoiceCallTospecifiedNumbers,refillAmount,refillRange
0,PA00000001,99999,300,NaN,20%,"['영상통화', '부가통화']"


In [182]:
eng_colnames = [
    'includedVoiceCall',
    'includedVideoOrValueAddedCall',
    'includedVoiceCallTospecifiedNumbers',
    'refillAmountRatio',
    'refillRange'
]

kor_colnames = [
    '음성통화제공량',
    '영상및부가통화제공량',
    '지정번호통화제공량',
    '리필비율한도',
    '리필대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [183]:
# 리스트형으로 변환
voice_table["refillRange"] = voice_table["refillRange"].apply(lambda x: type_cast(x))

# refillAmount 정규화
voice_table["refillAmountRatio"] = voice_table["refillAmount"].apply(lambda x: int(x.replace("%", ""))*0.01 if pd.notna(x) else None)

In [184]:
voice_table.iloc[109]["refillRange"]

[]

In [ ]:
with driver.session() as session:
    for _, row in voice_table.iterrows():
        # 속성값을 한글 컬럼명으로 변환
        properties = {}
        for col in eng_colnames:
            if type(row[col]) == list:
                properties[kor_cols_map[col]] = row[col]
            elif pd.isna(row[col]):
                # NaN 값을 'null'로 변환 -> 이렇게 해도 되는지 확인 필요 (neo4j에서는 NaN 또는 None을 속성값으로 넣을 수 없음)
                properties[kor_cols_map[col]] = 'null'
            else:
                properties[kor_cols_map[col]] = row[col]
        
        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (v:음성통화 {
                음성통화제공량: $음성통화제공량,
                영상및부가통화제공량: $영상및부가통화제공량,
                지정번호통화제공량: $지정번호통화제공량,
                리필비율한도: $리필비율한도,
                리필대상: $리필대상
            })
            MERGE (p)-[:제공]->(v)
            """,
            product_id=row['pmProductID'],
            음성통화제공량=properties['음성통화제공량'],
            영상및부가통화제공량=properties['영상및부가통화제공량'],
            지정번호통화제공량=properties['지정번호통화제공량'],
            리필비율한도=properties['리필비율한도'],
            리필대상=properties['리필대상']
        )

### SMS
- 문자메시지 제공량 관련 정보
- Label: 문자메시지
- 관계
  - 요금제 -[:제공]-> 문자메시지

In [ ]:
# SMS 데이터 로드
sms_table = pd.read_csv(os.path.join(DATA_DIR, "SMS.csv"))
sms_table.head(1)

,pmProductID,includedText,textRange
0,PA00000001,99999,[]


In [189]:
# 컬럼명 매핑 정의
eng_colnames = [
    'includedText',
    'textRange'
]

kor_colnames = [
    '문자제공량',
    '문자대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 그래프에 저장 -> 'textRange'는 빈 리스트만 존재해서 제외
with driver.session() as session:
    for _, row in sms_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (s:문자메시지 {
                문자제공량: $문자제공량
            })
            MERGE (p)-[:제공]->(s)
            """,
            product_id=row['pmProductID'],
            문자제공량=properties['문자제공량']
        )

### DATA
- 데이터 제공량 관련
- Label: 데이터용량
- 관계
  - 요금제 -[:제공]-> 데이터용량

In [191]:
data_table = pd.read_csv(os.path.join(DATA_DIR, "DATA.csv"))
data_table.head(1)

,pmProductID,includedData,includedDataForSharingAndTethering,includedMVoIP,appliedSpeed,seniorDataExceedAvailable,generalDataExceedAvailable,dataRefillAmount,dataRefillCouponGiftingAvailability,maximumShareAmount,dataGiftReceivingAvailability
0,PA00000001,15.0,15.0,15.0,1.0,N,N,15.0,Y,2GB,Y


In [193]:
# 컬럼명 매핑 정의
eng_colnames = [
    'includedData', 'includedDataForSharingAndTethering',
    'includedMVoIP', 'appliedSpeed', 'seniorDataExceedAvailable',
    'generalDataExceedAvailable', 'dataRefillAmount',
    'dataRefillCouponGiftingAvailability', 'maximumShareAmount',
    'dataGiftReceivingAvailability'
]

kor_colnames = [
    '기본제공데이터용량', '기본제공데이터중공유가능용량',
    '기본제공데이터중mvoip용량', '데이터소진후데이터제공속도', '시니어대상데이터소진후최대금액및속도제한적용',
    '데이터소진후최대금액및속도제한적용', '데이터리필가능용량',
    '데이터리필쿠폰선물가능여부', '최대데이터선물가능용량',
    '데이터선물받기가능여부'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 그래프에 저장
with driver.session() as session:
    for _, row in data_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (d:데이터용량 {
            기본제공데이터용량: $기본제공데이터용량,
            기본제공데이터중공유가능용량: $기본제공데이터중공유가능용량,
            기본제공데이터중mvoip용량: $기본제공데이터중mvoip용량,
            데이터소진후데이터제공속도: $데이터소진후데이터제공속도,
            시니어대상데이터소진후최대금액및속도제한적용: $시니어대상데이터소진후최대금액및속도제한적용,
            데이터소진후최대금액및속도제한적용: $데이터소진후최대금액및속도제한적용,
            데이터리필가능용량: $데이터리필가능용량,
            데이터리필쿠폰선물가능여부: $데이터리필쿠폰선물가능여부,
            최대데이터선물가능용량: $최대데이터선물가능용량,
            데이터선물받기가능여부: $데이터선물받기가능여부
            })
            MERGE (p)-[:제공]->(d)
            """,
            product_id=row['pmProductID'],
            기본제공데이터용량=properties.get('기본제공데이터용량', 'null'),
            기본제공데이터중공유가능용량=properties.get('기본제공데이터중공유가능용량', 'null'),
            기본제공데이터중mvoip용량=properties.get('기본제공데이터중mvoip용량', 'null'),
            데이터소진후데이터제공속도=properties.get('데이터소진후데이터제공속도', 'null'),
            시니어대상데이터소진후최대금액및속도제한적용=properties.get('시니어대상데이터소진후최대금액및속도제한적용', 'null'),
            데이터소진후최대금액및속도제한적용=properties.get('데이터소진후최대금액및속도제한적용', 'null'),
            데이터리필가능용량=properties.get('데이터리필가능용량', 'null'),
            데이터리필쿠폰선물가능여부=properties.get('데이터리필쿠폰선물가능여부', 'null'),
            최대데이터선물가능용량=properties.get('최대데이터선물가능용량', 'null'),
            데이터선물받기가능여부=properties.get('데이터선물받기가능여부', 'null')
        )